In [ ]:
#0 NAIVE BASELINE - config.
#like-for-like against the structured pipeline: SAME 10 filings, SAME embedding model,
#SAME gold questions. the only thing that changes is how the text is cut up - fixed
#220-word windows instead of structure-aware chunks.
#
#deliberate deviation from build_closeout_plan Step 4: it said to extract with pymupdf
#"to reproduce the naive pipeline honestly". we extract from the SAME HTML instead, so
#extraction is held constant and chunking is the only variable.
CHUNK_WORDS   = 220
CHUNK_OVERLAP = 35
EMBED_MODEL   = "BAAI/bge-small-en-v1.5"

GOLD_PATH     = "../data/gold/gold_set_60_v3.jsonl"
STRUCT_PATH   = "../data/gold/all_chunks_v2.jsonl"
RESULTS_DIR   = "../results"

FILINGS = {
    "aapl_FY2025_10K": "APPLE 10-K FY2025",   "amzn_FY2025_10K": "AMAZON 10-K FY2025",
    "dell_FY2026_10K": "DELL 10-K FY2026",    "goog_FY2025_10K": "ALPHABET 10-K FY2025",
    "meta_FY2025_10K": "META 10-K FY2025",    "msft_FY2026_10K": "MICROSOFT 10-K FY2026",
    "nvda_FY2024_10K": "NVIDIA 10-K FY2024",  "nvda_FY2025_10K": "NVIDIA 10-K FY2025",
    "nvda_FY2026_10K": "NVIDIA 10-K FY2026",  "pltr_FY2025_10K": "PALANTIR 10-K FY2025",
}

In [ ]:
#1 parse + chunk. flatten each filing's HTML to one long string, then slide a fixed
#220-word window over it with 35 words of overlap. no headings, no tables, no section
#tracking - a chunk boundary can land mid-sentence or mid-table-row.
import json, os, re, warnings, glob
import numpy as np
warnings.filterwarnings("ignore")
from bs4 import BeautifulSoup


def chunk_by_words(text, size=CHUNK_WORDS, overlap=CHUNK_OVERLAP):
    words = text.split()
    out, i = [], 0
    while i < len(words):
        out.append(" ".join(words[i:i + size]))
        i += size - overlap
    return out


naive_chunks, naive_doc = [], []
for key, doc in FILINGS.items():
    html = open(f"../data/raw/{key}.html", encoding="utf-8", errors="ignore").read()
    soup = BeautifulSoup(html, "lxml")
    for t in soup(["script", "style"]):
        t.decompose()
    text = re.sub(r"\s+", " ", soup.get_text(" ", strip=True))
    cs = chunk_by_words(text)
    naive_chunks.extend(cs)
    naive_doc.extend([doc] * len(cs))
    print(f"  {key:18} {len(text.split()):>8,} words -> {len(cs):>5} chunks")

print(f"\nnaive corpus: {len(naive_chunks)} chunks "
      f"({CHUNK_WORDS}w / {CHUNK_OVERLAP} overlap)")

In [ ]:
#2 gold mapping. gold_chunks are indices into the STRUCTURED corpus; naive chunks have
#different boundaries, so those indices mean nothing here. closeout Step 4 recommended
#answer-string matching, which needs an `answer` field the gold set does not have yet.
#
#instead we map by content: both corpora come from the same HTML, so a gold chunk and
#the naive chunk covering the same region share their NUMBERS, and numbers survive
#re-serialisation exactly. same property the v2 gold remap relied on.
#
#acceptance is relative to the best overlap actually achievable for each gold chunk,
#with an absolute floor. a fixed cut would fail on large structured tables that span
#several 220-word windows, which would deflate recall for a measurement reason rather
#than a retrieval one.
gold = [json.loads(l) for l in open(GOLD_PATH, encoding="utf-8") if l.strip()]
struct = [json.loads(l) for l in open(STRUCT_PATH, encoding="utf-8") if l.strip()]

NUM = re.compile(r"\d[\d,]*(?:\.\d+)?")
REL_TO_BEST, MIN_ABS, SHINGLE_N, SHINGLE_MIN = 0.90, 0.25, 8, 0.30

def nums(t):
    return {x.replace(",", "").rstrip(".") for x in NUM.findall(t)}

def shingles(t, n=SHINGLE_N):
    w = re.sub(r"[^a-z0-9 ]", " ", t.lower()).split()
    return {" ".join(w[i:i + n]) for i in range(max(0, len(w) - n + 1))}

def body(t):
    return t.split("\n\n", 1)[1] if "\n\n" in t else t

REFS = sorted({i for q in gold for i in q["gold_chunks"]})
chunk_nums = [nums(c) for c in naive_chunks]
by_doc = {}
for j, d in enumerate(naive_doc):
    by_doc.setdefault(d, []).append(j)

covers, unmapped, bests = {}, [], []
for gi in REFS:
    sc = struct[gi]
    cand = by_doc[sc["doc"]]
    gn = nums(body(sc["text"]))
    if gn:
        fr = [(len(gn & chunk_nums[j]) / len(gn), j) for j in cand]
        best = max(f for f, _ in fr) if fr else 0.0
        hit = [j for f, j in fr if f >= max(MIN_ABS, REL_TO_BEST * best) and f > 0]
    else:
        gs = shingles(body(sc["text"]))
        fr = [(len(gs & shingles(naive_chunks[j])) / len(gs), j) for j in cand] if gs else []
        best = max(f for f, _ in fr) if fr else 0.0
        hit = [j for f, j in fr if f >= max(SHINGLE_MIN, REL_TO_BEST * best) and f > 0]
    bests.append(best)
    covers[gi] = hit
    if not hit:
        unmapped.append(gi)

print(f"gold chunks mapped {len(REFS)-len(unmapped)}/{len(REFS)}"
      f"   mean best-overlap {np.mean(bests):.2f}"
      f"   mean covering chunks {np.mean([len(v) for v in covers.values() if v]):.1f}")
if unmapped:
    print(f"UNMAPPED {unmapped} - these can never score as hits, so recall below is"
          f" an under-estimate")

In [ ]:
#3 embed + retrieve. dense only - no BM25, no reranker, no decomposition.
#this is the floor the rest of the project is measured against.
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(EMBED_MODEL)
naive_emb = model.encode(naive_chunks, normalize_embeddings=True, show_progress_bar=True)
print(naive_emb.shape)

_qv = {}

def naive_search(query, k=10):
    if query not in _qv:
        _qv[query] = model.encode([query], normalize_embeddings=True)[0]
    return list(np.argsort(naive_emb @ _qv[query])[::-1][:k])

In [ ]:
#4 score. same definitions as evaluate.ipynb: recall@k = at least one required piece of
#evidence retrieved; all-gold@k = every required piece retrieved.
def evaluate_naive(ks=(1, 5, 10, 20)):
    answerable = [q for q in gold if q["gold_chunks"]]
    hard = [q for q in answerable if q["type"] == "hard"]
    easy = [q for q in answerable if q["type"] == "easy"]
    top = {q["id"]: naive_search(q["question"], max(ks)) for q in answerable}

    m = {"n_answerable": len(answerable), "n_hard": len(hard), "n_easy": len(easy)}
    for k in ks:
        got = {qid: set(v[:k]) for qid, v in top.items()}
        for label, subset in (("", answerable), ("easy_", easy), ("hard_", hard)):
            hits = sum(1 for q in subset
                       if any(got[q["id"]] & set(covers[g]) for g in q["gold_chunks"]))
            m[f"{label}recall@{k}"] = round(hits / len(subset), 3)
            m[f"{label}recall@{k}_hits"] = hits
        allg = sum(1 for q in hard
                   if all(got[q["id"]] & set(covers[g]) for g in q["gold_chunks"]))
        m[f"hard_all_gold@{k}"] = round(allg / len(hard), 3)
        m[f"hard_all_gold@{k}_hits"] = allg
    return m, top

naive_metrics, naive_top = evaluate_naive()
for k in (1, 5, 10, 20):
    print(f"recall@{k:<3} {naive_metrics[f'recall@{k}']:.3f} "
          f"({naive_metrics[f'recall@{k}_hits']}/{naive_metrics['n_answerable']})"
          f"    all-gold@{k:<3} {naive_metrics[f'hard_all_gold@{k}']:.3f} "
          f"({naive_metrics[f'hard_all_gold@{k}_hits']}/{naive_metrics['n_hard']})")

In [ ]:
#5 compare against the structured pipeline, and save.
import pandas as pd
from datetime import date

rows = [{"run": "naive_dense", "chunks": len(naive_chunks),
         **{f"r@{k}": naive_metrics[f"recall@{k}"] for k in (1, 5, 10, 20)},
         "ag@10": naive_metrics["hard_all_gold@10"],
         "ag@20": naive_metrics["hard_all_gold@20"]}]

for p in sorted(glob.glob(f"{RESULTS_DIR}/v2_*.json")):
    d = json.load(open(p))
    if d.get("config_name") == "refusal":
        continue
    m = d["metrics"]
    rows.append({"run": f"structured_{d['config_name']}", "chunks": d["n_chunks"],
                 **{f"r@{k}": m[f"recall@{k}"] for k in (1, 5, 10, 20)},
                 "ag@10": m["hard_all_gold@10"], "ag@20": m["hard_all_gold@20"]})

print(pd.DataFrame(rows).set_index("run").to_string())

payload = {
    "version": "naive", "gold_version": "v3", "config_name": "dense",
    "date": str(date.today()), "n_chunks": len(naive_chunks),
    "config": {"method": "fixed-window words, dense only",
               "chunk_words": CHUNK_WORDS, "overlap_words": CHUNK_OVERLAP,
               "embedding_model": EMBED_MODEL, "extraction": "HTML get_text",
               "scoring": "gold chunks mapped to naive chunks by numeric fingerprint",
               "unmapped_gold_chunks": len(unmapped)},
    "metrics": naive_metrics,
}
with open(f"{RESULTS_DIR}/naive_dense.json", "w") as f:
    json.dump(payload, f, indent=2)
print(f"\nsaved {RESULTS_DIR}/naive_dense.json")

In [ ]:
#6 failure read. which questions the naive pipeline misses at k=10, and what it
#returned instead.
def naive_failures(k=10, limit=6):
    shown = 0
    for q in gold:
        if not q["gold_chunks"]:
            continue
        got = set(naive_top[q["id"]][:k])
        if any(got & set(covers[g]) for g in q["gold_chunks"]):
            continue
        print(f"MISS [{q['id']}] {q['question'][:70]}")
        print(f"   needed any of: {[covers[g][:3] for g in q['gold_chunks']]}")
        for j in list(got)[:2]:
            print(f"   got [{j}] {naive_doc[j]}: {naive_chunks[j][:110]}")
        print()
        shown += 1
        if shown >= limit:
            break

naive_failures()